In [2]:
import time
import subprocess
import warnings
from dataclasses import dataclass, field
from typing import Optional
import psutil

try:
    import wmi
    _WMI_AVAILABLE = True
except ImportError:
    _WMI_AVAILABLE = False
    warnings.warn("WMI not found. Falling back to psutil-only mode.", RuntimeWarning)

print("Cell 1 OK")

Cell 1 OK


In [3]:
THERMAL_IDLE_GPU  = 40.0
THERMAL_LIMIT_GPU = 80.0
THERMAL_IDLE_CPU  = 45.0
THERMAL_LIMIT_CPU = 90.0
BATTERY_CRITICAL  = 15.0
BATTERY_LOW       = 30.0
WEIGHT_THERMAL    = 0.55
WEIGHT_BATTERY    = 0.45
EPSILON_THRESHOLD = 0.4

print("Cell 2 OK")

Cell 2 OK


In [4]:
@dataclass
class ThermalState:
    gpu_temp_c:    Optional[float] = None
    cpu_temp_c:    Optional[float] = None
    thermal_score: float           = 1.0

@dataclass
class BatteryState:
    percent:       float         = 100.0
    plugged_in:    bool          = True
    secs_left:     Optional[int] = None
    battery_score: float         = 1.0

@dataclass
class HardwareSnapshot:
    thermal:     ThermalState = field(default_factory=ThermalState)
    battery:     BatteryState = field(default_factory=BatteryState)
    cpu_usage:   float        = 0.0
    ram_free_gb: float        = 0.0
    efficiency:  float        = 1.0
    timestamp:   float        = field(default_factory=time.time)

def _clamp(value, low=0.0, high=1.0):
    return max(low, min(high, value))

def _compute_thermal_score(gpu_c, cpu_c):
    scores = []
    if gpu_c is not None:
        scores.append(1.0 - _clamp((gpu_c - THERMAL_IDLE_GPU) / (THERMAL_LIMIT_GPU - THERMAL_IDLE_GPU)))
    if cpu_c is not None:
        scores.append(1.0 - _clamp((cpu_c - THERMAL_IDLE_CPU) / (THERMAL_LIMIT_CPU - THERMAL_IDLE_CPU)))
    return min(scores) if scores else 0.8

def _compute_battery_score(percent, plugged_in):
    if plugged_in:
        return 1.0
    if percent >= 100:
        return 1.0
    elif percent >= BATTERY_LOW:
        return 0.5 + 0.5 * ((percent - BATTERY_LOW) / (100.0 - BATTERY_LOW))
    elif percent >= BATTERY_CRITICAL:
        return 0.5 * ((percent - BATTERY_CRITICAL) / (BATTERY_LOW - BATTERY_CRITICAL))
    else:
        return 0.0

def _compute_efficiency(thermal_score, battery_score):
    return round(_clamp(WEIGHT_THERMAL * thermal_score + WEIGHT_BATTERY * battery_score), 4)

print("Cell 3 OK")

Cell 3 OK


In [5]:
class TelemetryMonitor:

    def __init__(self, use_wmi=True):
        self._wmi_conn = None
        if use_wmi and _WMI_AVAILABLE:
            try:
                self._wmi_conn = wmi.WMI()
            except Exception as e:
                msg = f"WMI connection failed: {e}. Using psutil fallback."
                warnings.warn(msg)

    def snapshot(self):
        battery    = self._read_battery()
        thermal    = self._read_thermal()
        cpu_pct    = self._read_cpu()
        ram_gb     = self._read_ram()
        efficiency = _compute_efficiency(thermal.thermal_score, battery.battery_score)
        return HardwareSnapshot(
            thermal=thermal, battery=battery,
            cpu_usage=cpu_pct, ram_free_gb=ram_gb,
            efficiency=efficiency, timestamp=time.time(),
        )

    def report(self):
        s   = self.snapshot()
        ts  = time.strftime('%H:%M:%S', time.localtime(s.timestamp))
        gpu = s.thermal.gpu_temp_c if s.thermal.gpu_temp_c is not None else "N/A"
        cpu = s.thermal.cpu_temp_c if s.thermal.cpu_temp_c is not None else "N/A"
        src = 'AC (Plugged In)' if s.battery.plugged_in else 'DC (Battery)'
        tl  = self._format_time(s.battery.secs_left)
        lines = [
            "=" * 56,
            "  MODULE B — Hardware Telemetry Snapshot",
            "=" * 56,
            f"  Timestamp     : {ts}",
            f"  E(h) Score    : {s.efficiency:.4f}   (epsilon threshold input)",
            "",
            "  [ Thermal ]",
            f"    GPU Temp    : {gpu}°C",
            f"    CPU Temp    : {cpu}°C",
            f"    Thermal Score : {s.thermal.thermal_score:.4f}",
            "",
            "  [ Battery ]",
            f"    Level       : {s.battery.percent:>5.1f}%",
            f"    Source      : {src}",
            f"    Time Left   : {tl}",
            f"    Battery Score : {s.battery.battery_score:.4f}",
            "",
            "  [ System Load ]",
            f"    CPU Usage   : {s.cpu_usage:>5.1f}%",
            f"    Free RAM    : {s.ram_free_gb:>5.2f} GB",
            "=" * 56,
        ]
        return "\n".join(lines)

    def _read_battery(self):
        batt = psutil.sensors_battery()
        if batt is None:
            return BatteryState(percent=100.0, plugged_in=True, secs_left=None, battery_score=1.0)
        percent    = float(batt.percent)
        plugged_in = bool(batt.power_plugged)
        secs_left  = int(batt.secsleft) if batt.secsleft not in (None, -1, -2) else None
        if secs_left is None and self._wmi_conn is not None:
            try:
                for b in self._wmi_conn.Win32_Battery():
                    if b.EstimatedRunTime and b.EstimatedRunTime < 1440:
                        secs_left = int(b.EstimatedRunTime) * 60
                        break
            except Exception:
                pass
        return BatteryState(
            percent=percent, plugged_in=plugged_in,
            secs_left=secs_left,
            battery_score=_compute_battery_score(percent, plugged_in),
        )

    def _read_thermal(self):
        gpu_temp = self._read_gpu_temp_via_nvidia_smi()
        cpu_temp = self._read_cpu_temp_via_wmi()
        return ThermalState(
            gpu_temp_c=gpu_temp,
            cpu_temp_c=cpu_temp,
            thermal_score=_compute_thermal_score(gpu_temp, cpu_temp),
        )

    def _read_gpu_temp_via_nvidia_smi(self):
        try:
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=temperature.gpu", "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=3,
            )
            if result.returncode == 0:
                return float(result.stdout.strip().split("\n")[0])
        except (FileNotFoundError, ValueError, subprocess.TimeoutExpired):
            pass
        return None

    def _read_cpu_temp_via_wmi(self):
        if self._wmi_conn is None:
            return None
        try:
            readings = []
            for zone in self._wmi_conn.MSAcpi_ThermalZoneTemperature():
                celsius = (zone.CurrentTemperature / 10.0) - 273.15
                if 0 < celsius < 120:
                    readings.append(celsius)
            return max(readings) if readings else None
        except Exception:
            return None

    def _read_cpu(self):
        return psutil.cpu_percent(interval=1)

    def _read_ram(self):
        return round(psutil.virtual_memory().available / (1024 ** 3), 2)

    @staticmethod
    def _format_time(secs):
        if secs is None:
            return "N/A"
        h, m = divmod(secs // 60, 60)
        return f"{h}h {m:02d}m"

print("Cell 4 OK")

Cell 4 OK


In [6]:
monitor = TelemetryMonitor()
print(monitor.report())

snap = monitor.snapshot()
if snap.efficiency < EPSILON_THRESHOLD:
    print(f"\n→ M_small  (E(h)={snap.efficiency} < ε={EPSILON_THRESHOLD})")
elif snap.thermal.gpu_temp_c is not None and snap.thermal.gpu_temp_c > THERMAL_LIMIT_GPU:
    print(f"\n→ M_small  (GPU thermal override: {snap.thermal.gpu_temp_c}°C)")
else:
    print(f"\n→ M_large eligible  (E(h)={snap.efficiency})")

  MODULE B — Hardware Telemetry Snapshot
  Timestamp     : 18:34:28
  E(h) Score    : 0.5225   (epsilon threshold input)

  [ Thermal ]
    GPU Temp    : 54.0°C
    CPU Temp    : N/A°C
    Thermal Score : 0.6500

  [ Battery ]
    Level       :  26.0%
    Source      : DC (Battery)
    Time Left   : 0h 33m
    Battery Score : 0.3667

  [ System Load ]
    CPU Usage   :   3.9%
    Free RAM    :  5.49 GB

→ M_large eligible  (E(h)=0.5225)


In [7]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [14]:
import re
import time
import warnings
import numpy as np
from dataclasses import dataclass
from typing import Optional
from sentence_transformers import SentenceTransformer

print("Cell 6 OK")

Cell 6 OK


In [15]:
try:
    from sentence_transformers import SentenceTransformer
    _ST_AVAILABLE = True
except ImportError:
    _ST_AVAILABLE = False
    warnings.warn(
        "sentence-transformers not found. "
        "Install with: pip install sentence-transformers",
        RuntimeWarning,
    )
 
print("Cell 1 OK — imports loaded.")
 

Cell 1 OK — imports loaded.


In [16]:
TAU_THRESHOLD = 0.5
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EASY_ANCHORS = [
    "What is the capital of France?",
    "What time is it?",
    "Tell me a joke.",
    "What does CPU stand for?",
    "How do I say hello in Spanish?",
    "What is 2 + 2?",
    "Define the word 'ephemeral'.",
    "Who wrote Romeo and Juliet?",
]
 
HARD_ANCHORS = [
    "Derive the backpropagation algorithm for a multi-layer neural network.",
    "Explain the mathematical relationship between entropy and mutual information.",
    "Write a research paper introduction on transformer attention mechanisms.",
    "Implement a red-black tree with full insertion and deletion logic.",
    "Compare the epistemological foundations of rationalism and empiricism.",
    "Prove that the square root of 2 is irrational using contradiction.",
    "Design a distributed system architecture for real-time fraud detection.",
    "Analyze the geopolitical implications of semiconductor supply chain risks.",
]
 
print("Cell 2 OK — constants loaded.")
 

Cell 2 OK — constants loaded.


In [17]:
@dataclass
class ComplexityResult:
    """
    Output of the Semantic Classifier for a single prompt.
 
    Fields
    ------
    prompt : str
        The original input prompt.
    score : float
        S(p) — semantic complexity score in [0.0, 1.0].
        0.0 = trivially simple, 1.0 = maximally complex.
    label : str
        Binary routing label: 'EASY' or 'HARD'.
        Derived by comparing score against TAU_THRESHOLD.
    tau : float
        The threshold τ used for this classification.
    inference_ms : float
        Time taken to compute the embedding in milliseconds.
        PAPER: Used to report Module A latency overhead in Table 1.
    method : str
        Always 'embedding' for this module. Included for
        compatibility with future hybrid classifier versions.
    """
    prompt:       str
    score:        float
    label:        str
    tau:          float
    inference_ms: float
    method:       str = "embedding"
 
    def __str__(self):
        bar_len  = 30
        filled   = int(self.score * bar_len)
        bar      = "█" * filled + "░" * (bar_len - filled)
        return (
            f"\n  Prompt      : {self.prompt[:60]}{'...' if len(self.prompt)>60 else ''}\n"
            f"  S(p) Score  : {self.score:.4f}  [{bar}]\n"
            f"  Label       : {self.label}  (τ = {self.tau})\n"
            f"  Latency     : {self.inference_ms:.1f} ms\n"
        )
 
 
def _cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Compute cosine similarity between two vectors.
 
    PAPER: S(p) uses cosine similarity because it measures the
    angular distance between embeddings independent of magnitude,
    making it robust to prompt length variation.
 
    cos(a, b) = (a · b) / (||a|| * ||b||)
    """
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)
 
 
def _mean_embedding(model: SentenceTransformer,
                    sentences: list) -> np.ndarray:
    """
    Compute the mean embedding of a list of anchor sentences.
 
    PAPER: Using the centroid of multiple anchor embeddings
    produces a more stable reference vector than a single
    anchor, reducing sensitivity to phrasing variation.
    """
    embeddings = model.encode(sentences, convert_to_numpy=True)
    return np.mean(embeddings, axis=0)
 
 
def _compute_complexity_score(prompt_emb:  np.ndarray,
                               easy_anchor: np.ndarray,
                               hard_anchor: np.ndarray) -> float:
    """
    Compute S(p) using contrastive cosine similarity.
 
    PAPER FORMULA:
        sim_hard = cosine(p, hard_anchor)
        sim_easy = cosine(p, easy_anchor)
        S(p) = (sim_hard + 1) / (sim_hard + sim_easy + 2)
 
    The +1 and +2 terms normalize cosine similarity from [-1,1]
    to [0,1] before computing the ratio, ensuring S(p) ∈ [0,1].
 
    Interpretation:
        S(p) → 1.0  when p is more similar to hard anchors
        S(p) → 0.0  when p is more similar to easy anchors
        S(p) = 0.5  when equally similar to both (boundary)
    """
    sim_hard = _cosine_similarity(prompt_emb, hard_anchor)
    sim_easy = _cosine_similarity(prompt_emb, easy_anchor)
 
    # Normalize from [-1,1] cosine range to [0,1]
    score = (sim_hard + 1.0) / (sim_hard + sim_easy + 2.0)
    return round(float(score), 4)
 
 
print("Cell 3 OK — data classes and helpers loaded.")

Cell 3 OK — data classes and helpers loaded.


In [18]:
class SemanticClassifier:
    """
    Module A: Semantic Complexity Classifier.
 
    Uses sentence-transformer embeddings and contrastive anchor
    comparison to estimate how complex a prompt is, returning
    S(p) ∈ [0.0, 1.0] and a binary EASY/HARD label.
 
    The model is loaded once at initialization and reused for
    all subsequent classify() calls, keeping per-query latency
    under 10ms on CPU after the first call.
 
    Usage in Module C (Routing Engine):
    ------------------------------------
        classifier = SemanticClassifier()
        result = classifier.classify("Explain quantum entanglement")
 
        if result.score < TAU_THRESHOLD:
            route_to(M_small)
        else:
            route_to(M_large)
    """
 
    def __init__(self,
                 model_name: str = EMBEDDING_MODEL_NAME,
                 tau: float = TAU_THRESHOLD):
        """
        Parameters
        ----------
        model_name : str
            HuggingFace model identifier for sentence-transformers.
            Default: 'all-MiniLM-L6-v2' (22M params, 384-dim).
        tau : float
            Complexity threshold τ. Prompts with S(p) >= tau
            are labelled HARD and routed to M_large.
        """
        self.tau = tau
        self._model = None
 
        print(f"  Loading embedding model: {model_name} ...")
        t0 = time.time()
 
        if not _ST_AVAILABLE:
            raise ImportError(
                "sentence-transformers is required. "
                "Run: pip install sentence-transformers"
            )
 
        self._model = SentenceTransformer(model_name)
 
        # Pre-compute anchor centroids once at init time.
        # PAPER: This amortizes the cost of anchor embedding
        # over the entire session (not per query).
        print("  Computing anchor embeddings ...")
        self._easy_anchor = _mean_embedding(self._model, EASY_ANCHORS)
        self._hard_anchor = _mean_embedding(self._model, HARD_ANCHORS)
 
        elapsed = (time.time() - t0) * 1000
        print(f"  SemanticClassifier ready in {elapsed:.0f} ms.\n")
 
    # ── Public API ────────────────────────────────────────────
 
    def classify(self, prompt: str) -> ComplexityResult:
        """
        Classify a single prompt and return its complexity result.
 
        Parameters
        ----------
        prompt : str
            The raw user input to classify.
 
        Returns
        -------
        ComplexityResult
            Contains S(p) score, EASY/HARD label, and latency.
        """
        t0 = time.time()
 
        prompt_emb = self._model.encode(
            [self._preprocess(prompt)],
            convert_to_numpy=True,
        )[0]
 
        score = _compute_complexity_score(
            prompt_emb   = prompt_emb,
            easy_anchor  = self._easy_anchor,
            hard_anchor  = self._hard_anchor,
        )
 
        inference_ms = (time.time() - t0) * 1000
        label        = "HARD" if score >= self.tau else "EASY"
 
        return ComplexityResult(
            prompt       = prompt,
            score        = score,
            label        = label,
            tau          = self.tau,
            inference_ms = inference_ms,
        )
 
    def classify_batch(self, prompts: list) -> list:
        """
        Classify a list of prompts efficiently using batch encoding.
 
        PAPER: Batch encoding amortizes GPU/CPU overhead when
        evaluating the classifier on the test set of 1,000 prompts
        used in the energy measurement experiments (Section 5).
 
        Returns
        -------
        list of ComplexityResult
        """
        t0           = time.time()
        cleaned      = [self._preprocess(p) for p in prompts]
        embeddings   = self._model.encode(cleaned, convert_to_numpy=True)
        total_ms     = (time.time() - t0) * 1000
        per_ms       = total_ms / max(len(prompts), 1)
 
        results = []
        for prompt, emb in zip(prompts, embeddings):
            score = _compute_complexity_score(emb, self._easy_anchor, self._hard_anchor)
            label = "HARD" if score >= self.tau else "EASY"
            results.append(ComplexityResult(
                prompt       = prompt,
                score        = score,
                label        = label,
                tau          = self.tau,
                inference_ms = per_ms,
            ))
        return results
 
    # ── Private helpers ───────────────────────────────────────
 
    @staticmethod
    def _preprocess(prompt: str) -> str:
        """
        Lightweight prompt normalization before embedding.
 
        Steps:
          1. Strip leading/trailing whitespace
          2. Collapse multiple spaces/newlines to single space
          3. Truncate to 512 characters (model input limit)
 
        PAPER: Preprocessing is kept minimal to avoid altering
        the semantic content that the embedding captures.
        """
        prompt = prompt.strip()
        prompt = re.sub(r'\s+', ' ', prompt)
        return prompt[:512]
 
 
print("Cell 4 OK — SemanticClassifier loaded.")
 

Cell 4 OK — SemanticClassifier loaded.


In [19]:
print("=" * 56)
print("  Initializing Module A (Semantic Classifier) ...")
print("=" * 56)
classifier = SemanticClassifier()
 
print("=" * 56)
print("  Initializing Module B (Telemetry Monitor) ...")
print("=" * 56)
monitor = TelemetryMonitor()   # Already loaded from Module B cells
snap    = monitor.snapshot()
print(monitor.report())
 
# ── Step 2: Define test prompts ───────────────────────────────
test_prompts = [
    "What is 5 times 6?",
    "How do neural networks learn?",
    "Derive the gradient of the softmax cross-entropy loss.",
    "Tell me a fun fact.",
    "Compare BERT and GPT architectures in detail.",
    "What is Python?",
    "Explain the CAP theorem and its implications for distributed databases.",
]
 
# ── Step 3: Classify and route each prompt ───────────────────
print("=" * 56)
print("  D(p, h) — Full Routing Decisions")
print(f"  E(h) = {snap.efficiency:.4f}   ε = {EPSILON_THRESHOLD}")
print(f"  τ    = {TAU_THRESHOLD}")
print("=" * 56)
 
for prompt in test_prompts:
    result = classifier.classify(prompt)
 
    # ── Routing Logic D(p, h) ─────────────────────────────────
    # PAPER FORMULA:
    #   D(p,h) = M_small  if  S(p) < τ  OR  E(h) < ε
    #            M_large  otherwise
    # Hard thermal override also forces M_small regardless of S(p)
    thermal_override = (
        snap.thermal.gpu_temp_c is not None and
        snap.thermal.gpu_temp_c > THERMAL_LIMIT_GPU
    )
 
    if thermal_override:
        decision = "M_small  ⚠ THERMAL OVERRIDE"
        reason   = f"GPU={snap.thermal.gpu_temp_c}°C > {THERMAL_LIMIT_GPU}°C"
    elif snap.efficiency < EPSILON_THRESHOLD:
        decision = "M_small  ⚡ LOW EFFICIENCY"
        reason   = f"E(h)={snap.efficiency} < ε={EPSILON_THRESHOLD}"
    elif result.score < TAU_THRESHOLD:
        decision = "M_small  ✓ EASY QUERY"
        reason   = f"S(p)={result.score} < τ={TAU_THRESHOLD}"
    else:
        decision = "M_large  ✓ HARD QUERY"
        reason   = f"S(p)={result.score} >= τ={TAU_THRESHOLD}, E(h)={snap.efficiency} >= ε"
 
    print(f"\n  Prompt  : {prompt[:55]}{'...' if len(prompt)>55 else ''}")
    print(f"  S(p)    : {result.score:.4f}  [{result.label}]  ({result.inference_ms:.1f} ms)")
    print(f"  Route   : {decision}")
    print(f"  Reason  : {reason}")
 
print("\n" + "=" * 56)
print("  Module A + B integration complete.")
print("=" * 56)

  Initializing Module A (Semantic Classifier) ...
  Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3156.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing anchor embeddings ...
  SemanticClassifier ready in 19494 ms.

  Initializing Module B (Telemetry Monitor) ...
  MODULE B — Hardware Telemetry Snapshot
  Timestamp     : 18:36:55
  E(h) Score    : 0.5887   (epsilon threshold input)

  [ Thermal ]
    GPU Temp    : 47.0°C
    CPU Temp    : N/A°C
    Thermal Score : 0.8250

  [ Battery ]
    Level       :  24.0%
    Source      : DC (Battery)
    Time Left   : 0h 27m
    Battery Score : 0.3000

  [ System Load ]
    CPU Usage   :   3.7%
    Free RAM    :  5.12 GB
  D(p, h) — Full Routing Decisions
  E(h) = 0.5887   ε = 0.4
  τ    = 0.5

  Prompt  : What is 5 times 6?
  S(p)    : 0.4549  [EASY]  (56.3 ms)
  Route   : M_small  ✓ EASY QUERY
  Reason  : S(p)=0.4549 < τ=0.5

  Prompt  : How do neural networks learn?
  S(p)    : 0.5498  [HARD]  (25.5 ms)
  Route   : M_large  ✓ HARD QUERY
  Reason  : S(p)=0.5498 >= τ=0.5, E(h)=0.5887 >= ε

  Prompt  : Derive the gradient of the softmax cross-entropy loss.
  S(p)    : 0.5742  [HARD] 

In [20]:
import time
import json
import warnings
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

print("Cell 11 OK")

Cell 11 OK


In [21]:
MODEL_SMALL          = "qwen-1.5b"
MODEL_LARGE          = "qwen-7b"
ROUTER_TAU           = 0.5
ROUTER_EPSILON       = 0.4
OVERRIDE_GPU_TEMP_C  = 80.0
OVERRIDE_BATTERY_PCT = 15.0
REASON_THERMAL_OVERRIDE = "THERMAL_OVERRIDE"
REASON_BATTERY_OVERRIDE = "BATTERY_OVERRIDE"
REASON_LOW_EFFICIENCY   = "LOW_EFFICIENCY"
REASON_EASY_QUERY       = "EASY_QUERY"
REASON_HARD_QUERY       = "HARD_QUERY"

print("Cell 12 OK")

Cell 12 OK


In [22]:
class ModelChoice(Enum):
    """
    Enum representing the two inference models.
    PAPER: M_small = lightweight model (Qwen-1.5B)
           M_large = high-accuracy model (Qwen-7B)
    Using an enum prevents string-comparison bugs in Module D.
    """
    SMALL = "M_small"
    LARGE = "M_large"
 
 
@dataclass
class RoutingDecision:
    """
    The complete output of one routing evaluation.
    This is the object passed to Module D (Inference Executor).
 
    Fields
    ------
    model : ModelChoice
        Which model to use: ModelChoice.SMALL or ModelChoice.LARGE
    reason : str
        Reason code explaining why this decision was made.
        One of the REASON_* constants defined above.
    prompt : str
        The original user prompt being routed.
    complexity_score : float
        S(p) from Module A — semantic complexity in [0.0, 1.0]
    efficiency_score : float
        E(h) from Module B — hardware efficiency in [0.0, 1.0]
    gpu_temp_c : float or None
        GPU temperature at time of decision (for logging).
    battery_pct : float
        Battery percentage at time of decision (for logging).
    tau : float
        The τ threshold used in this decision.
    epsilon : float
        The ε threshold used in this decision.
    timestamp : float
        Unix timestamp of the routing decision.
    decision_ms : float
        Time taken by the routing engine in milliseconds.
        PAPER: Reported as routing overhead in Table 1.
    """
    model:            ModelChoice
    reason:           str
    prompt:           str
    complexity_score: float
    efficiency_score: float
    gpu_temp_c:       Optional[float]
    battery_pct:      float
    tau:              float
    epsilon:          float
    timestamp:        float = field(default_factory=time.time)
    decision_ms:      float = 0.0
 
    def to_dict(self) -> dict:
        """
        Serialize to dict for CSV/JSON logging.
        PAPER: Used to generate the evaluation dataset in Section 5.
        """
        return {
            "timestamp":        self.timestamp,
            "model":            self.model.value,
            "reason":           self.reason,
            "complexity_score": self.complexity_score,
            "efficiency_score": self.efficiency_score,
            "gpu_temp_c":       self.gpu_temp_c,
            "battery_pct":      self.battery_pct,
            "tau":              self.tau,
            "epsilon":          self.epsilon,
            "decision_ms":      self.decision_ms,
            "prompt_length":    len(self.prompt),
        }
 
    def __str__(self) -> str:
        ts  = time.strftime('%H:%M:%S', time.localtime(self.timestamp))
        sep = "=" * 56
        return (
            f"\n{sep}\n"
            f"  MODULE C — Routing Decision\n"
            f"{sep}\n"
            f"  Timestamp   : {ts}\n"
            f"  Prompt      : {self.prompt[:55]}{'...' if len(self.prompt)>55 else ''}\n"
            f"\n"
            f"  S(p)        : {self.complexity_score:.4f}  (τ = {self.tau})\n"
            f"  E(h)        : {self.efficiency_score:.4f}  (ε = {self.epsilon})\n"
            f"  GPU Temp    : {self.gpu_temp_c if self.gpu_temp_c else 'N/A'}°C\n"
            f"  Battery     : {self.battery_pct:.1f}%\n"
            f"\n"
            f"  ► Decision  : {self.model.value}\n"
            f"  ► Reason    : {self.reason}\n"
            f"  ► Latency   : {self.decision_ms:.2f} ms\n"
            f"{sep}"
        )
 
 
@dataclass
class RouterStats:
    """
    Cumulative statistics across all routing decisions in a session.
    PAPER: These counters feed directly into the efficiency metrics
           reported in Section 5 (e.g., % of queries served by M_small,
           energy saved per 1,000 tokens).
    """
    total_decisions:   int   = 0
    small_model_count: int   = 0
    large_model_count: int   = 0
    thermal_overrides: int   = 0
    battery_overrides: int   = 0
    total_decision_ms: float = 0.0
 
    @property
    def small_model_pct(self) -> float:
        if self.total_decisions == 0:
            return 0.0
        return round(100 * self.small_model_count / self.total_decisions, 2)
 
    @property
    def avg_decision_ms(self) -> float:
        if self.total_decisions == 0:
            return 0.0
        return round(self.total_decision_ms / self.total_decisions, 3)
 
    def __str__(self) -> str:
        return (
            f"\n  [ Router Session Statistics ]\n"
            f"    Total Decisions   : {self.total_decisions}\n"
            f"    M_small Routed    : {self.small_model_count} ({self.small_model_pct}%)\n"
            f"    M_large Routed    : {self.large_model_count}\n"
            f"    Thermal Overrides : {self.thermal_overrides}\n"
            f"    Battery Overrides : {self.battery_overrides}\n"
            f"    Avg Decision Time : {self.avg_decision_ms} ms\n"
        )
 
 
print("Cell 13 OK — data classes loaded.")
 

Cell 13 OK — data classes loaded.


In [23]:
class DynamicRouter:
    """
    Module C: Dynamic Routing Engine.
 
    Combines the outputs of Module A (SemanticClassifier) and
    Module B (TelemetryMonitor) to implement the routing formula:
 
        D(p, h) = M_small  if  S(p) < τ  OR  E(h) < ε
                  M_large  otherwise
 
    Hard overrides (thermal, battery) are checked first and
    bypass the soft routing logic entirely.
 
    The router maintains a RouterStats object across the session
    for use in the paper's evaluation section.
    """
 
    def __init__(self,
                 classifier:  "SemanticClassifier",
                 monitor:     "TelemetryMonitor",
                 tau:         float = ROUTER_TAU,
                 epsilon:     float = ROUTER_EPSILON):
        """
        Parameters
        ----------
        classifier : SemanticClassifier
            Module A instance — provides S(p).
        monitor : TelemetryMonitor
            Module B instance — provides E(h) and hardware state.
        tau : float
            Semantic complexity threshold τ.
        epsilon : float
            Hardware efficiency threshold ε.
        """
        self.classifier = classifier
        self.monitor    = monitor
        self.tau        = tau
        self.epsilon    = epsilon
        self.stats      = RouterStats()
        self._log       = []           # In-memory decision log for analysis
 
        print(f"  DynamicRouter ready.  τ={self.tau}  ε={self.epsilon}\n")
 
    # ── Public API ────────────────────────────────────────────
 
    def route(self, prompt: str) -> RoutingDecision:
        """
        Main entry point. Given a prompt, return a RoutingDecision.
 
        Execution order (PAPER — Section 3.2):
          1. Capture hardware snapshot  → E(h)
          2. Check hard overrides       → thermal / battery
          3. Compute semantic score     → S(p)
          4. Apply routing formula      → D(p, h)
          5. Update session statistics
          6. Return RoutingDecision
 
        Parameters
        ----------
        prompt : str
            Raw user input.
 
        Returns
        -------
        RoutingDecision
        """
        t0 = time.time()
 
        # ── Step 1: Hardware snapshot ─────────────────────────
        # Hardware is sampled BEFORE semantic classification so
        # that a thermal spike is caught even for easy queries.
        snap = self.monitor.snapshot()
 
        gpu_temp    = snap.thermal.gpu_temp_c
        battery_pct = snap.battery.percent
        efficiency  = snap.efficiency
 
        # ── Step 2: Hard overrides ────────────────────────────
        # These bypass S(p) entirely to protect hardware.
        # PAPER: Hard overrides are non-negotiable safety
        #        constraints, not soft routing preferences.
        if gpu_temp is not None and gpu_temp > OVERRIDE_GPU_TEMP_C:
            model  = ModelChoice.SMALL
            reason = REASON_THERMAL_OVERRIDE
            # Still compute S(p) for logging, but skip it for routing
            sp = self.classifier.classify(prompt).score
            return self._finalize(
                model, reason, prompt, sp,
                efficiency, gpu_temp, battery_pct, t0,
                override=True,
            )
 
        if battery_pct < OVERRIDE_BATTERY_PCT:
            model  = ModelChoice.SMALL
            reason = REASON_BATTERY_OVERRIDE
            sp = self.classifier.classify(prompt).score
            return self._finalize(
                model, reason, prompt, sp,
                efficiency, gpu_temp, battery_pct, t0,
                override=True,
            )
 
        # ── Step 3: Semantic classification ──────────────────
        # Only reached if no hard override triggered.
        complexity = self.classifier.classify(prompt)
        sp         = complexity.score
 
        # ── Step 4: Routing formula D(p, h) ──────────────────
        # PAPER FORMULA:
        #   D(p,h) = M_small if S(p) < τ OR E(h) < ε
        #            M_large otherwise
        if efficiency < self.epsilon:
            model  = ModelChoice.SMALL
            reason = REASON_LOW_EFFICIENCY
        elif sp < self.tau:
            model  = ModelChoice.SMALL
            reason = REASON_EASY_QUERY
        else:
            model  = ModelChoice.LARGE
            reason = REASON_HARD_QUERY
 
        return self._finalize(
            model, reason, prompt, sp,
            efficiency, gpu_temp, battery_pct, t0,
        )
 
    def route_batch(self, prompts: list) -> list:
        """
        Route a list of prompts efficiently.
        Hardware is sampled once for the entire batch.
        PAPER: Used in the 1,000-prompt energy evaluation (Section 5).
 
        Returns
        -------
        list of RoutingDecision
        """
        # Single hardware snapshot for the whole batch
        snap        = self.monitor.snapshot()
        gpu_temp    = snap.thermal.gpu_temp_c
        battery_pct = snap.battery.percent
        efficiency  = snap.efficiency
 
        # Batch classify all prompts at once (faster than one-by-one)
        complexity_results = self.classifier.classify_batch(prompts)
 
        decisions = []
        for prompt, complexity in zip(prompts, complexity_results):
            t0 = time.time()
            sp = complexity.score
 
            # Apply same override + routing logic per prompt
            if gpu_temp is not None and gpu_temp > OVERRIDE_GPU_TEMP_C:
                model, reason = ModelChoice.SMALL, REASON_THERMAL_OVERRIDE
            elif battery_pct < OVERRIDE_BATTERY_PCT:
                model, reason = ModelChoice.SMALL, REASON_BATTERY_OVERRIDE
            elif efficiency < self.epsilon:
                model, reason = ModelChoice.SMALL, REASON_LOW_EFFICIENCY
            elif sp < self.tau:
                model, reason = ModelChoice.SMALL, REASON_EASY_QUERY
            else:
                model, reason = ModelChoice.LARGE, REASON_HARD_QUERY
 
            decisions.append(self._finalize(
                model, reason, prompt, sp,
                efficiency, gpu_temp, battery_pct, t0,
            ))
 
        return decisions
 
    def get_stats(self) -> RouterStats:
        """Return cumulative session statistics."""
        return self.stats
 
    def export_log(self) -> list:
        """
        Return all routing decisions as a list of dicts.
        PAPER: Serialize with json.dumps() or pandas.DataFrame()
               to generate Table 4 and Figure 3 in the paper.
        """
        return [d.to_dict() for d in self._log]
 
    def reset_stats(self):
        """Reset session statistics and log (e.g., between experiments)."""
        self.stats = RouterStats()
        self._log  = []
 
    # ── Private helpers ───────────────────────────────────────
 
    def _finalize(self,
                  model:       ModelChoice,
                  reason:      str,
                  prompt:      str,
                  sp:          float,
                  efficiency:  float,
                  gpu_temp:    Optional[float],
                  battery_pct: float,
                  t0:          float,
                  override:    bool = False) -> RoutingDecision:
        """
        Build RoutingDecision, update stats, append to log, return.
        Centralizing this avoids duplicating stat-update logic.
        """
        decision_ms = (time.time() - t0) * 1000
 
        decision = RoutingDecision(
            model            = model,
            reason           = reason,
            prompt           = prompt,
            complexity_score = sp,
            efficiency_score = efficiency,
            gpu_temp_c       = gpu_temp,
            battery_pct      = battery_pct,
            tau              = self.tau,
            epsilon          = self.epsilon,
            timestamp        = time.time(),
            decision_ms      = decision_ms,
        )
 
        # ── Update session statistics ─────────────────────────
        self.stats.total_decisions   += 1
        self.stats.total_decision_ms += decision_ms
 
        if model == ModelChoice.SMALL:
            self.stats.small_model_count += 1
        else:
            self.stats.large_model_count += 1
 
        if reason == REASON_THERMAL_OVERRIDE:
            self.stats.thermal_overrides += 1
        if reason == REASON_BATTERY_OVERRIDE:
            self.stats.battery_overrides += 1
 
        self._log.append(decision)
        return decision
 
 
print("Cell 14 OK — DynamicRouter loaded.")
 

Cell 14 OK — DynamicRouter loaded.


In [24]:
print("=" * 56)
print("  Initializing Module C (Dynamic Router) ...")
print("=" * 56)
 
router = DynamicRouter(
    classifier = classifier,   # From Module A (Cell 9)
    monitor    = monitor,      # From Module B (Cell 4)
    tau        = ROUTER_TAU,
    epsilon    = ROUTER_EPSILON,
)
 
# ── Test prompts covering all routing paths ───────────────────
test_prompts = [
    # Expected EASY → M_small
    "What is the capital of France?",
    "Tell me a joke.",
    "What time is it in Tokyo?",
 
    # Expected HARD → M_large
    "Derive the backpropagation equations for an LSTM network.",
    "Compare the architectural differences between BERT and GPT-3.",
    "Explain the CAP theorem and its trade-offs in distributed systems.",
 
    # Borderline cases
    "How does attention work in transformers?",
    "What is machine learning?",
]
 
print("\n  Running routing decisions for all test prompts...\n")
 
for prompt in test_prompts:
    decision = router.route(prompt)
    print(decision)
 
# ── Session statistics summary ────────────────────────────────
print("\n" + "=" * 56)
print("  SESSION STATISTICS (for paper evaluation section)")
print("=" * 56)
print(router.get_stats())
 
# ── Export log preview ────────────────────────────────────────
print("  Log export preview (first 2 entries):")
log = router.export_log()
for entry in log[:2]:
    print(f"    {json.dumps(entry, indent=2)}")
 
print("\n  Module C integration complete.")
print("  All three modules (A + B + C) are operational.")

  Initializing Module C (Dynamic Router) ...
  DynamicRouter ready.  τ=0.5  ε=0.4


  Running routing decisions for all test prompts...


  MODULE C — Routing Decision
  Timestamp   : 18:38:18
  Prompt      : What is the capital of France?

  S(p)        : 0.3940  (τ = 0.5)
  E(h)        : 0.6013  (ε = 0.4)
  GPU Temp    : 45.0°C
  Battery     : 23.0%

  ► Decision  : M_small
  ► Reason    : EASY_QUERY
  ► Latency   : 1136.48 ms

  MODULE C — Routing Decision
  Timestamp   : 18:38:19
  Prompt      : Tell me a joke.

  S(p)        : 0.4231  (τ = 0.5)
  E(h)        : 0.6013  (ε = 0.4)
  GPU Temp    : 45.0°C
  Battery     : 23.0%

  ► Decision  : M_small
  ► Reason    : EASY_QUERY
  ► Latency   : 1147.77 ms

  MODULE C — Routing Decision
  Timestamp   : 18:38:21
  Prompt      : What time is it in Tokyo?

  S(p)        : 0.4344  (τ = 0.5)
  E(h)        : 0.6013  (ε = 0.4)
  GPU Temp    : 45.0°C
  Battery     : 23.0%

  ► Decision  : M_small
  ► Reason    : EASY_QUERY
  ► Latency   : 1159.3

In [25]:
import time
import json
import os
import warnings
from dataclasses import dataclass, field
from typing import Optional

try:
    from llama_cpp import Llama
    _LLAMA_AVAILABLE = True
except ImportError:
    _LLAMA_AVAILABLE = False
    warnings.warn("llama-cpp-python not found. Running in SIMULATION MODE.", RuntimeWarning)

print("Cell 16 OK")

Cell 16 OK


In [26]:
PATH_MODEL_SMALL = "models/qwen1.5-1.8b-chat-q4_k_m.gguf"
PATH_MODEL_LARGE = "models/qwen-7b-chat-q4_k_m.gguf"
 
# ── VRAM / RAM allocation ─────────────────────────────────────
# n_gpu_layers controls how many transformer layers are offloaded
# to the GPU. -1 = offload all layers.
# PAPER: M_small is fully offloaded (-1) to keep it "hot" in VRAM.
#        M_large uses partial offload (N_GPU_LAYERS_LARGE) to
#        avoid VRAM overflow on RTX 50-series 16GB cards.
N_GPU_LAYERS_SMALL = -1     # Full GPU offload for M_small
N_GPU_LAYERS_LARGE = 20     # Partial offload for M_large
 
# ── Context window ────────────────────────────────────────────
# PAPER: Both models use a 2048-token context window to maintain
#        a fair comparison. Larger windows increase VRAM usage.
CONTEXT_LENGTH = 2048
 
# ── Generation parameters ─────────────────────────────────────
# PAPER: Sampling parameters are fixed across both models to
#        isolate the effect of model size on output quality.
MAX_NEW_TOKENS  = 512
TEMPERATURE     = 0.7
TOP_P           = 0.9
REPEAT_PENALTY  = 1.1
 
# ── Energy estimation constants ───────────────────────────────
# PAPER: Estimated TDP values for RTX 50-series GPU (Section 5.2)
# These are used to compute energy-per-token approximations.
# Actual measurements should use a hardware power meter (e.g., NVML).
TDP_SMALL_WATTS = 40.0    # Estimated GPU watts during M_small inference
TDP_LARGE_WATTS = 120.0   # Estimated GPU watts during M_large inference
 
print("Cell 17 OK — Module D constants loaded.")

Cell 17 OK — Module D constants loaded.


In [27]:
@dataclass
class InferenceResult:
    """
    Output of one complete inference call.
    Returned by InferenceWrapper.generate() to the application layer.
 
    Fields
    ------
    prompt : str
        The original user prompt.
    response : str
        The model's generated text response.
    model_used : str
        Which model was used: MODEL_SMALL or MODEL_LARGE.
    routing_reason : str
        The REASON_* code from Module C's RoutingDecision.
    tokens_generated : int
        Number of tokens in the response.
        PAPER: Used to compute tokens/sec and energy/token metrics.
    tokens_per_sec : float
        Generation throughput. PAPER: Reported in Table 1.
    energy_estimate_j : float
        Estimated energy consumed in Joules.
        Formula: energy_j = (TDP_watts * generation_time_s)
        PAPER: Used in Figure 4 (energy per 1,000 tokens).
    total_ms : float
        End-to-end latency including routing overhead.
    context_length : int
        Number of tokens in the conversation history at this turn.
    """
    prompt:            str
    response:          str
    model_used:        str
    routing_reason:    str
    tokens_generated:  int   = 0
    tokens_per_sec:    float = 0.0
    energy_estimate_j: float = 0.0
    total_ms:          float = 0.0
    context_length:    int   = 0
 
    def __str__(self) -> str:
        sep = "=" * 56
        return (
            f"\n{sep}\n"
            f"  MODULE D — Inference Result\n"
            f"{sep}\n"
            f"  Model Used   : {self.model_used}\n"
            f"  Reason       : {self.routing_reason}\n"
            f"  Tokens       : {self.tokens_generated}\n"
            f"  Speed        : {self.tokens_per_sec:.1f} tok/s\n"
            f"  Energy       : {self.energy_estimate_j:.4f} J\n"
            f"  Latency      : {self.total_ms:.1f} ms\n"
            f"  Context Len  : {self.context_length} tokens\n"
            f"\n"
            f"  Prompt   : {self.prompt[:55]}{'...' if len(self.prompt)>55 else ''}\n"
            f"  Response : {self.response[:200]}{'...' if len(self.response)>200 else ''}\n"
            f"{sep}"
        )
 
    def to_dict(self) -> dict:
        """Serialize for CSV/JSON logging in the paper's dataset."""
        return {
            "model_used":        self.model_used,
            "routing_reason":    self.routing_reason,
            "tokens_generated":  self.tokens_generated,
            "tokens_per_sec":    self.tokens_per_sec,
            "energy_estimate_j": self.energy_estimate_j,
            "total_ms":          self.total_ms,
            "context_length":    self.context_length,
            "prompt_length":     len(self.prompt),
        }
 
 
@dataclass
class ConversationTurn:
    """
    A single turn in the conversation history.
 
    PAPER: Context Handoff — when the router switches from
    M_small to M_large mid-conversation, the full history
    (all prior turns) is passed to M_large so it has complete
    context. This is the Context Handoff mechanism cited in
    Section 3.4 of the paper.
    """
    role:    str   # "user" or "assistant"
    content: str
    model:   str   # which model generated this turn
    tokens:  int   # approximate token count
 
 
@dataclass
class SessionStats:
    """
    Cumulative inference statistics for the session.
    PAPER: Feeds into Section 5 energy evaluation metrics.
    """
    total_calls:         int   = 0
    small_model_calls:   int   = 0
    large_model_calls:   int   = 0
    total_tokens:        int   = 0
    total_energy_j:      float = 0.0
    total_latency_ms:    float = 0.0
    context_switches:    int   = 0   # M_small → M_large transitions
 
    @property
    def energy_per_1k_tokens(self) -> float:
        """
        PAPER: Primary energy efficiency metric (Section 5.2).
        Formula: E_1k = (total_energy_j / total_tokens) * 1000
        """
        if self.total_tokens == 0:
            return 0.0
        return round((self.total_energy_j / self.total_tokens) * 1000, 4)
 
    @property
    def avg_latency_ms(self) -> float:
        if self.total_calls == 0:
            return 0.0
        return round(self.total_latency_ms / self.total_calls, 2)
 
    def __str__(self) -> str:
        return (
            f"\n  [ Inference Session Statistics ]\n"
            f"    Total Calls          : {self.total_calls}\n"
            f"    M_small Calls        : {self.small_model_calls}\n"
            f"    M_large Calls        : {self.large_model_calls}\n"
            f"    Total Tokens         : {self.total_tokens}\n"
            f"    Total Energy         : {self.total_energy_j:.4f} J\n"
            f"    Energy / 1K Tokens   : {self.energy_per_1k_tokens:.4f} J  ← Paper metric\n"
            f"    Avg Latency          : {self.avg_latency_ms:.2f} ms\n"
            f"    Context Switches     : {self.context_switches}\n"
        )
 
 
print("Cell 18 OK — Module D data classes loaded.")
 

Cell 18 OK — Module D data classes loaded.


In [28]:
class InferenceWrapper:
    """
    Module D: Inference Wrapper — The Executor.
 
    Manages model loading, KV-cache strategy, context handoff,
    and energy logging. Receives RoutingDecision from Module C
    and returns InferenceResult to the application.
 
    KV-Cache Strategy (PAPER Section 3.4):
    ----------------------------------------
    M_small is loaded at startup and kept resident in VRAM at
    all times (n_gpu_layers=-1). M_large is loaded on first use
    and kept resident with partial GPU offload. This "hot cache"
    strategy eliminates model-loading latency after the first call.
 
    Context Handoff (PAPER Section 3.4):
    ----------------------------------------
    The conversation history is stored as a list of
    ConversationTurn objects. When the router switches models,
    the full history is serialized into the new model's prompt
    format so no context is lost.
 
    Simulation Mode:
    ----------------------------------------
    If llama-cpp-python is not installed, the wrapper runs in
    simulation mode — returning plausible fake responses with
    realistic latency estimates. This allows the routing pipeline
    (Modules A+B+C+D) to be tested end-to-end without models.
    """
 
    def __init__(self,
                 path_small:       str = PATH_MODEL_SMALL,
                 path_large:       str = PATH_MODEL_LARGE,
                 n_gpu_small:      int = N_GPU_LAYERS_SMALL,
                 n_gpu_large:      int = N_GPU_LAYERS_LARGE,
                 context_length:   int = CONTEXT_LENGTH,
                 simulation_mode:  bool = False):
        """
        Parameters
        ----------
        path_small : str
            Path to the M_small GGUF model file.
        path_large : str
            Path to the M_large GGUF model file.
        n_gpu_small : int
            GPU layers for M_small (-1 = all).
        n_gpu_large : int
            GPU layers for M_large (partial offload).
        context_length : int
            Token context window size for both models.
        simulation_mode : bool
            If True, skip model loading and return simulated responses.
            Automatically set to True if llama-cpp-python is missing.
        """
        self.path_small      = path_small
        self.path_large      = path_large
        self.n_gpu_small     = n_gpu_small
        self.n_gpu_large     = n_gpu_large
        self.context_length  = context_length
        self.simulation_mode = simulation_mode or not _LLAMA_AVAILABLE
 
        # Model instances — loaded lazily on first use
        self._model_small: Optional[object] = None
        self._model_large: Optional[object] = None
 
        # Conversation history for context handoff
        self._history: list = []
        self._last_model: Optional[str] = None
 
        # Session statistics
        self.stats = SessionStats()
 
        if self.simulation_mode:
            print("  InferenceWrapper running in SIMULATION MODE.")
            print("  (Install llama-cpp-python + download GGUF models for real inference)\n")
        else:
            # Pre-load M_small at startup to keep it hot in VRAM
            print("  Pre-loading M_small into VRAM (KV-cache strategy) ...")
            self._load_small()
            print("  InferenceWrapper ready.\n")
 
    # ── Public API ────────────────────────────────────────────
 
    def generate(self,
                 prompt:   str,
                 decision: "RoutingDecision") -> InferenceResult:
        """
        Execute inference using the model specified in the decision.
 
        This is the main entry point called by the application after
        receiving a RoutingDecision from Module C.
 
        Parameters
        ----------
        prompt : str
            The user's raw input prompt.
        decision : RoutingDecision
            Output from Module C specifying which model to use.
 
        Returns
        -------
        InferenceResult
            Contains the response, token counts, energy estimate,
            and latency — all used in the paper's evaluation.
        """
        t0         = time.time()
        model_name = decision.model.value
 
        # ── Detect context switch ─────────────────────────────
        # PAPER: A context switch occurs when the router selects
        # a different model than the previous turn. We log these
        # to measure how often handoff overhead is incurred.
        if self._last_model is not None and self._last_model != model_name:
            self.stats.context_switches += 1
 
        # ── Build prompt with conversation history ────────────
        full_prompt = self._build_prompt(prompt, model_name)
 
        # ── Run inference ─────────────────────────────────────
        if self.simulation_mode:
            response, tokens = self._simulate_inference(prompt, model_name)
        else:
            response, tokens = self._run_inference(full_prompt, model_name)
 
        # ── Compute metrics ───────────────────────────────────
        total_ms   = (time.time() - t0) * 1000
        gen_secs   = total_ms / 1000.0
        tok_per_s  = tokens / gen_secs if gen_secs > 0 else 0.0
 
        # Energy estimate: E = TDP (W) * time (s)
        # PAPER: Section 5.2 — energy per token approximation.
        tdp       = TDP_SMALL_WATTS if "small" in model_name.lower() \
                    else TDP_LARGE_WATTS
        energy_j  = tdp * gen_secs
 
        # ── Update conversation history ───────────────────────
        self._history.append(ConversationTurn(
            role    = "user",
            content = prompt,
            model   = model_name,
            tokens  = len(prompt.split()),
        ))
        self._history.append(ConversationTurn(
            role    = "assistant",
            content = response,
            model   = model_name,
            tokens  = tokens,
        ))
        self._last_model = model_name
 
        # ── Update session stats ──────────────────────────────
        self.stats.total_calls      += 1
        self.stats.total_tokens     += tokens
        self.stats.total_energy_j   += energy_j
        self.stats.total_latency_ms += total_ms
 
        if "small" in model_name.lower():
            self.stats.small_model_calls += 1
        else:
            self.stats.large_model_calls += 1
 
        return InferenceResult(
            prompt            = prompt,
            response          = response,
            model_used        = model_name,
            routing_reason    = decision.reason,
            tokens_generated  = tokens,
            tokens_per_sec    = round(tok_per_s, 2),
            energy_estimate_j = round(energy_j, 6),
            total_ms          = round(total_ms, 2),
            context_length    = len(self._history),
        )
 
    def clear_history(self):
        """Reset conversation history. Call between sessions."""
        self._history    = []
        self._last_model = None
 
    def get_stats(self) -> SessionStats:
        """Return cumulative session statistics."""
        return self.stats
 
    def export_log(self) -> list:
        """Export conversation history as list of dicts."""
        return [
            {"role": t.role, "content": t.content,
             "model": t.model, "tokens": t.tokens}
            for t in self._history
        ]
 
    # ── Private: model loading ────────────────────────────────
 
    def _load_small(self):
        """
        Load M_small with full GPU offload (KV-cache hot strategy).
        PAPER: M_small is always resident in VRAM to eliminate
        cold-start latency for easy queries routed at high frequency.
        """
        if not os.path.exists(self.path_small):
            raise FileNotFoundError(
                f"M_small model not found at: {self.path_small}\n"
                f"Download a Qwen-1.5B GGUF from HuggingFace and update PATH_MODEL_SMALL."
            )
        self._model_small = Llama(
            model_path    = self.path_small,
            n_gpu_layers  = self.n_gpu_small,
            n_ctx         = self.context_length,
            verbose       = False,
        )
 
    def _load_large(self):
        """
        Load M_large with partial GPU offload on first use.
        PAPER: Lazy loading of M_large avoids occupying VRAM
        when the workload consists mostly of easy queries.
        """
        if not os.path.exists(self.path_large):
            raise FileNotFoundError(
                f"M_large model not found at: {self.path_large}\n"
                f"Download a Qwen-7B GGUF from HuggingFace and update PATH_MODEL_LARGE."
            )
        self._model_large = Llama(
            model_path    = self.path_large,
            n_gpu_layers  = self.n_gpu_large,
            n_ctx         = self.context_length,
            verbose       = False,
        )
 
    # ── Private: inference execution ─────────────────────────
 
    def _run_inference(self, full_prompt: str,
                       model_name: str) -> tuple:
        """
        Run real inference using the selected llama-cpp model.
 
        Returns
        -------
        (response_text, token_count)
        """
        if "small" in model_name.lower():
            if self._model_small is None:
                self._load_small()
            model = self._model_small
        else:
            if self._model_large is None:
                print("  Loading M_large on first use ...")
                self._load_large()
            model = self._model_large
 
        output = model(
            full_prompt,
            max_tokens     = MAX_NEW_TOKENS,
            temperature    = TEMPERATURE,
            top_p          = TOP_P,
            repeat_penalty = REPEAT_PENALTY,
            echo           = False,
        )
 
        response = output["choices"][0]["text"].strip()
        tokens   = output["usage"]["completion_tokens"]
        return response, tokens
 
    def _simulate_inference(self, prompt: str,
                            model_name: str) -> tuple:
        """
        Simulate inference without loading real models.
 
        Returns realistic latency and token estimates based on
        typical Qwen performance on RTX 50-series hardware.
        PAPER: Simulation mode is used for pipeline testing only.
              All numbers in the paper use real model inference.
        """
        import random
 
        if "small" in model_name.lower():
            # M_small: ~80 tok/s on RTX 4070
            tokens   = random.randint(40, 120)
            sim_secs = tokens / 80.0
        else:
            # M_large: ~25 tok/s on RTX 4070
            tokens   = random.randint(80, 300)
            sim_secs = tokens / 25.0
 
        time.sleep(sim_secs * 0.1)   # 10% of real time for demo
 
        response = (
            f"[SIMULATED {model_name.upper()} RESPONSE — "
            f"{tokens} tokens — "
            f"Install llama-cpp-python for real inference]\n"
            f"This is a placeholder response to: '{prompt[:40]}...'"
        )
        return response, tokens
 
    # ── Private: prompt building ──────────────────────────────
 
    def _build_prompt(self, prompt: str, model_name: str) -> str:
        """
        Build the full prompt including conversation history.
 
        PAPER: Context Handoff — the entire prior conversation
        is prepended to the new prompt when the model switches.
        This ensures M_large has full context when taking over
        from M_small mid-session.
 
        Uses the ChatML format used by Qwen models:
            <|im_start|>system
            You are a helpful assistant.
            <|im_end|>
            <|im_start|>user
            {user_message}
            <|im_end|>
            <|im_start|>assistant
        """
        system = "You are a helpful AI assistant."
        parts  = [f"<|im_start|>system\n{system}\n<|im_end|>"]
 
        # Append conversation history (last 10 turns max to manage context)
        for turn in self._history[-10:]:
            parts.append(
                f"<|im_start|>{turn.role}\n{turn.content}\n<|im_end|>"
            )
 
        # Append current user prompt
        parts.append(f"<|im_start|>user\n{prompt}\n<|im_end|>")
        parts.append("<|im_start|>assistant\n")
 
        return "\n".join(parts)
 
 
print("Cell 19 OK — InferenceWrapper loaded.")
 

Cell 19 OK — InferenceWrapper loaded.


In [29]:
print("=" * 56)
print("  FULL SYSTEM TEST — All 4 Modules")
print("  Dynamic Inference System")
print("=" * 56)
 
# ── Initialize Module D ───────────────────────────────────────
# simulation_mode=True because we don't have models downloaded yet.
# Set to False once you have the GGUF files in your models/ folder.
executor = InferenceWrapper(simulation_mode=True)
 
# ── Test conversation (mixed easy + hard queries) ─────────────
conversation = [
    "What is 10 times 7?",
    "Explain how transformer attention mechanisms work in detail.",
    "What is the capital of Germany?",
    "Derive the mathematical proof for the universal approximation theorem.",
    "Tell me a fun fact about space.",
    "Compare BERT vs GPT-3 architectures for NLP tasks.",
]
 
print("\n  Starting inference session...\n")
 
for prompt in conversation:
    # ── Module C: get routing decision ───────────────────────
    decision = router.route(prompt)
 
    # ── Module D: execute inference ───────────────────────────
    result = executor.generate(prompt, decision)
 
    # ── Print combined output ─────────────────────────────────
    print(result)
 
# ── Final statistics from all modules ────────────────────────
print("\n" + "=" * 56)
print("  FINAL SESSION STATISTICS")
print("=" * 56)
 
print("\n  [ Module C — Routing Stats ]")
print(router.get_stats())
 
print("\n  [ Module D — Inference Stats ]")
print(executor.get_stats())
 
# ── Energy efficiency summary ─────────────────────────────────
# PAPER: This is the primary result reported in Section 5.
inf_stats = executor.get_stats()
print("=" * 56)
print("  PAPER METRICS SUMMARY")
print("=" * 56)
print(f"  Energy / 1K Tokens : {inf_stats.energy_per_1k_tokens:.4f} J")
print(f"  M_small Usage      : {router.get_stats().small_model_pct}%")
print(f"  Context Switches   : {inf_stats.context_switches}")
print(f"  Avg Latency        : {inf_stats.avg_latency_ms:.2f} ms")
print("=" * 56)
print("\n  All 4 modules operational. System ready for evaluation.")
 

  FULL SYSTEM TEST — All 4 Modules
  Dynamic Inference System
  InferenceWrapper running in SIMULATION MODE.
  (Install llama-cpp-python + download GGUF models for real inference)


  Starting inference session...


  MODULE D — Inference Result
  Model Used   : M_small
  Reason       : EASY_QUERY
  Tokens       : 66
  Speed        : 790.1 tok/s
  Energy       : 3.3412 J
  Latency      : 83.5 ms
  Context Len  : 2 tokens

  Prompt   : What is 10 times 7?
  Response : [SIMULATED M_SMALL RESPONSE — 66 tokens — Install llama-cpp-python for real inference]
This is a placeholder response to: 'What is 10 times 7?...'

  MODULE D — Inference Result
  Model Used   : M_large
  Reason       : HARD_QUERY
  Tokens       : 98
  Speed        : 249.3 tok/s
  Energy       : 47.1770 J
  Latency      : 393.1 ms
  Context Len  : 4 tokens

  Prompt   : Explain how transformer attention mechanisms work in de...
  Response : [SIMULATED M_LARGE RESPONSE — 98 tokens — Install llama-cpp-python for real inferen

In [30]:
import subprocess
subprocess.run(["pip", "install", "huggingface_hub"], check=True)
print("Done")

Done


In [31]:
import os
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])  # set HF_TOKEN in your shell env, never hardcode it

In [32]:
# !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="Qwen/Qwen1.5-1.8B-Chat-GGUF",
	filename="qwen1_5-1_8b-chat-q2_k.gguf",
)

C:\Users\Abde\anaconda3\envs\qwen_env\Lib\site-packages\huggingface_hub\utils\_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from C:\Users\Abde\.cache\huggingface\hub\models--Qwen--Qwen1.5-1.8B-Chat-GGUF\snapshots\c6414b00c3ed52df27201a5386a971869a72496a\.\qwen1_5-1_8b-chat-q2_k.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.name str              = Qwen1.5-1.8B-Chat-AWQ-fp16
llama_model_loader: - kv   2:                          qwen2.block_count u32              = 24
llama_model_loader: - kv   3:                   

In [33]:
llm.create_chat_completion(
	messages = [
		{
			"role": "user",
			"content": "What is the capital of France?"
		}
	]
)

llama_perf_context_print:        load time =     542.41 ms
llama_perf_context_print: prompt eval time =     538.43 ms /    20 tokens (   26.92 ms per token,    37.15 tokens per second)
llama_perf_context_print:        eval time =     348.33 ms /     7 runs   (   49.76 ms per token,    20.10 tokens per second)
llama_perf_context_print:       total time =     899.91 ms /    27 tokens
llama_perf_context_print:    graphs reused =          6


{'id': 'chatcmpl-69402c16-adf6-42ef-9bf8-b26305d69a6b',
 'object': 'chat.completion',
 'created': 1776422396,
 'model': 'C:\\Users\\Abde\\.cache\\huggingface\\hub\\models--Qwen--Qwen1.5-1.8B-Chat-GGUF\\snapshots\\c6414b00c3ed52df27201a5386a971869a72496a\\.\\qwen1_5-1_8b-chat-q2_k.gguf',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'The capital of France is Paris.'},
   'logprobs': None,
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 20, 'completion_tokens': 7, 'total_tokens': 27}}

In [34]:
# !pip install llama-cpp-python

from llama_cpp import Llama

llm = Llama.from_pretrained(
	repo_id="RichardErkhov/Qwen_-_Qwen-7B-Chat-gguf",
	filename="Qwen-7B-Chat.IQ3_M.gguf",
)

llama_model_loader: loaded meta data with 20 key-value pairs and 259 tensors from C:\Users\Abde\.cache\huggingface\hub\models--RichardErkhov--Qwen_-_Qwen-7B-Chat-gguf\snapshots\00396105889b7d0887d858afd1be6a0c1258b0f7\.\Qwen-7B-Chat.IQ3_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen
llama_model_loader: - kv   1:                               general.name str              = Qwen
llama_model_loader: - kv   2:                        qwen.context_length u32              = 32768
llama_model_loader: - kv   3:                           qwen.block_count u32              = 32
llama_model_loader: - kv   4:                      qwen.embedding_length u32              = 4096
llama_model_loader: - kv   5:                   qwen.feed_forward_length u32              = 22016
llama_model_loader: - kv   6:                    

In [35]:
output = llm(
	"Once upon a time,",
	max_tokens=512,
	echo=True
)
print(output)

llama_perf_context_print:        load time =    1787.97 ms
llama_perf_context_print: prompt eval time =    1787.76 ms /     5 tokens (  357.55 ms per token,     2.80 tokens per second)
llama_perf_context_print:        eval time =   93854.33 ms /   382 runs   (  245.69 ms per token,     4.07 tokens per second)
llama_perf_context_print:       total time =   96150.51 ms /   387 tokens
llama_perf_context_print:    graphs reused =        380


{'id': 'cmpl-97ae28e0-8503-49b6-9439-a9a38516c3ff', 'object': 'text_completion', 'created': 1776422412, 'model': 'C:\\Users\\Abde\\.cache\\huggingface\\hub\\models--RichardErkhov--Qwen_-_Qwen-7B-Chat-gguf\\snapshots\\00396105889b7d0887d858afd1be6a0c1258b0f7\\.\\Qwen-7B-Chat.IQ3_M.gguf', 'choices': [{'text': "Once upon a time, in a land far, far away, there lived a princess named Isabella. She was known for her kindness, beauty, and grace, but she was also known for her love of adventure. Isabella had always dreamed of going on a great journey, but her parents had other plans for her. They wanted her to marry a prince who was good-looking and had a lot of money, but Isabella knew that wasn't the kind of life she wanted.\n\nOne day, a handsome prince named Jack came to the kingdom seeking a princess to marry him. Isabella was smitten by his charm and bravery, but she also knew that she had to be careful. She didn't want to just fall in love with someone without knowing him first. So, she

In [36]:
import subprocess
subprocess.run([
    "pip", "install", "llama-cpp-python",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
], check=True)
print("llama-cpp-python installed with CUDA support.")

llama-cpp-python installed with CUDA support.


In [37]:
import os
from llama_cpp import Llama

# Updated paths to match the filenames in your images
# Note: Ensure these files are located inside a "models" folder if using these paths
small_path = "models/qwen1_5-1_8b-chat-q2_k.gguf"
large_path = "models/Qwen-7B-Chat.IQ3_M.gguf"

small_ok = os.path.exists(small_path)
large_ok = os.path.exists(large_path)

print(f"M_small file found : {small_ok}")
print(f"M_large file found : {large_ok}")

if small_ok and large_ok:
    print("\nAll models ready. You can now run Cell 20 with simulation_mode=False")
    
    # Example of how you would initialize them based on your images:
    # llm_small = Llama.from_pretrained(
    #     repo_id="Qwen/Qwen1.5-1.8B-Chat-GGUF", 
    #     filename="qwen1_5-1_8b-chat-q2_k.gguf"
    # )
else:
    print("\nOne or more files missing. Re-run the download steps above.")
    print(f"Expected Small: {small_path}")
    print(f"Expected Large: {large_path}")

M_small file found : False
M_large file found : False

One or more files missing. Re-run the download steps above.
Expected Small: models/qwen1_5-1_8b-chat-q2_k.gguf
Expected Large: models/Qwen-7B-Chat.IQ3_M.gguf


In [38]:
import os
from huggingface_hub import hf_hub_download

def check_and_get_path(repo, filename):
    try:
        # This looks in your cache; if not found, it downloads it.
        # It returns the absolute path to the file.
        path = hf_hub_download(repo_id=repo, filename=filename)
        return path
    except Exception as e:
        return None

# Defining the models from your images
small_repo = "Qwen/Qwen1.5-1.8B-Chat-GGUF"
small_file = "qwen1_5-1_8b-chat-q2_k.gguf"

large_repo = "RichardErkhov/Qwen_-_Qwen-7B-Chat-gguf"
large_file = "Qwen-7B-Chat.IQ3_M.gguf"

# Verify and get paths
path_small = check_and_get_path(small_repo, small_file)
path_large = check_and_get_path(large_repo, large_file)

print(f"M_small file found : {path_small is not None}")
print(f"M_large file found : {path_large is not None}")

if path_small and path_large:
    print("\n**All models ready.** You can now run the simulation.")
    print(f"Small model path: {path_small}")
    print(f"Large model path: {path_large}")
else:
    print("\n**One or more files missing.**")
    print("Please ensure you have an active internet connection to download them.")

M_small file found : True
M_large file found : True

**All models ready.** You can now run the simulation.
Small model path: C:\Users\Abde\.cache\huggingface\hub\models--Qwen--Qwen1.5-1.8B-Chat-GGUF\snapshots\c6414b00c3ed52df27201a5386a971869a72496a\qwen1_5-1_8b-chat-q2_k.gguf
Large model path: C:\Users\Abde\.cache\huggingface\hub\models--RichardErkhov--Qwen_-_Qwen-7B-Chat-gguf\snapshots\00396105889b7d0887d858afd1be6a0c1258b0f7\Qwen-7B-Chat.IQ3_M.gguf


In [39]:
from llama_cpp import Llama

# Use the variables from your previous cell (path_small and path_large)
# If this is a new cell, ensure path_small and path_large are defined.

print("Loading Large Model...")
llm_large = Llama(
    model_path=path_large,
    n_ctx=2048,      # Context window
    n_threads=8      # Adjust based on your CPU cores
)

print("Loading Small Model...")
llm_small = Llama(
    model_path=path_small,
    n_ctx=2048,
    n_threads=8
)

print("\n**Models loaded successfully!**")

llama_model_loader: loaded meta data with 20 key-value pairs and 259 tensors from C:\Users\Abde\.cache\huggingface\hub\models--RichardErkhov--Qwen_-_Qwen-7B-Chat-gguf\snapshots\00396105889b7d0887d858afd1be6a0c1258b0f7\Qwen-7B-Chat.IQ3_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen
llama_model_loader: - kv   1:                               general.name str              = Qwen
llama_model_loader: - kv   2:                        qwen.context_length u32              = 32768
llama_model_loader: - kv   3:                           qwen.block_count u32              = 32
llama_model_loader: - kv   4:                      qwen.embedding_length u32              = 4096
llama_model_loader: - kv   5:                   qwen.feed_forward_length u32              = 22016
llama_model_loader: - kv   6:                      

Loading Large Model...


llama_model_loader: - kv  16:                tokenizer.ggml.bos_token_id u32              = 151643
llama_model_loader: - kv  17:                tokenizer.ggml.eos_token_id u32              = 151643
llama_model_loader: - kv  18:            tokenizer.ggml.unknown_token_id u32              = 151643
llama_model_loader: - kv  19:               general.quantization_version u32              = 2
llama_model_loader: - type  f32:   97 tensors
llama_model_loader: - type q4_K:   68 tensors
llama_model_loader: - type q6_K:    1 tensors
llama_model_loader: - type iq3_s:   93 tensors
print_info: file format = GGUF V3 (latest)
print_info: file type   = IQ3_S mix - 3.66 bpw
print_info: file size   = 3.61 GiB (4.01 BPW) 
init_tokenizer: initializing tokenizer for type 2
load: 0 unused tokens
load: control token: 151728 '<|extra_82|>' is not marked as EOG
load: control token: 151837 '<|extra_191|>' is not marked as EOG
load: control token: 151707 '<|extra_61|>' is not marked as EOG
load: control token: 1

Loading Small Model...


llama_model_loader: - kv   2:                          qwen2.block_count u32              = 24
llama_model_loader: - kv   3:                       qwen2.context_length u32              = 32768
llama_model_loader: - kv   4:                     qwen2.embedding_length u32              = 2048
llama_model_loader: - kv   5:                  qwen2.feed_forward_length u32              = 5504
llama_model_loader: - kv   6:                 qwen2.attention.head_count u32              = 16
llama_model_loader: - kv   7:              qwen2.attention.head_count_kv u32              = 16
llama_model_loader: - kv   8:     qwen2.attention.layer_norm_rms_epsilon f32              = 0.000001
llama_model_loader: - kv   9:                       qwen2.rope.freq_base f32              = 1000000.000000
llama_model_loader: - kv  10:                qwen2.use_parallel_residual bool             = true
llama_model_loader: - kv  11:                       tokenizer.ggml.model str              = gpt2
llama_model_loader: -


**Models loaded successfully!**


CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'qwen2.attention.head_count': '16', 'general.name': 'Qwen1.5-1.8B-Chat-AWQ-fp16', 'general.architecture': 'qwen2', 'qwen2.block_count': '24', 'qwen2.context_length': '32768', 'qwen2.attention.head_count_kv': '16', 'qwen2.embedding_length': '2048', 'general.quantization_version': '2', 'tokenizer.ggml.bos_token_id': '151643', 'qwen2.feed_forward_length': '5504', 'qwen2.attention.layer_norm_rms_epsilon': '0.000001', 'tokenizer.ggml.padding_token_id': '151643', 'qwen2.rope.freq_base': '1000000.000000', 'qwen2.use_parallel_residual': 'true', 'tokenizer.ggml.model': 'gpt2', 'general.file_type': '10', 'tokenizer.ggml.eos_token_id': '151645', 'tokenizer.chat_template': "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %

In [40]:
# Test the large model
response = llm_large(
    "Question: What is the capital of France? Answer:",
    max_tokens=32,
    stop=["\n"],
    echo=True
)

print(response["choices"][0]["text"])

llama_perf_context_print:        load time =    2287.77 ms
llama_perf_context_print: prompt eval time =    2282.64 ms /    11 tokens (  207.51 ms per token,     4.82 tokens per second)
llama_perf_context_print:        eval time =    3419.83 ms /    14 runs   (  244.27 ms per token,     4.09 tokens per second)
llama_perf_context_print:       total time =    5734.60 ms /    25 tokens
llama_perf_context_print:    graphs reused =         13


Question: What is the capital of France? Answer: Tokyo is the capital of Japan. Paris is the capital of France.


In [41]:
import os
from llama_cpp import Llama

# Updated paths to match the filenames in your images
# Note: Ensure these files are located inside a "models" folder if using these paths
small_path = "models/qwen1_5-1_8b-chat-q2_k.gguf"
large_path = "models/Qwen-7B-Chat.IQ3_M.gguf"

small_ok = os.path.exists(small_path)
large_ok = os.path.exists(large_path)

print(f"M_small file found : {small_ok}")
print(f"M_large file found : {large_ok}")

if small_ok and large_ok:
    print("\nAll models ready. You can now run Cell 20 with simulation_mode=False")
    
    # Example of how you would initialize them based on your images:
    # llm_small = Llama.from_pretrained(
    #     repo_id="Qwen/Qwen1.5-1.8B-Chat-GGUF", 
    #     filename="qwen1_5-1_8b-chat-q2_k.gguf"
    # )
else:
    print("\nOne or more files missing. Re-run the download steps above.")
    print(f"Expected Small: {small_path}")
    print(f"Expected Large: {large_path}")

M_small file found : False
M_large file found : False

One or more files missing. Re-run the download steps above.
Expected Small: models/qwen1_5-1_8b-chat-q2_k.gguf
Expected Large: models/Qwen-7B-Chat.IQ3_M.gguf


In [42]:
# --- MODULE C & D: SIMPLE ROUTING ENGINE ---

def run_smart_inference(user_prompt, battery_percent, is_complex_task):
    """
    Decides which model to use and returns the result.
    """
    # 1. SET THRESHOLDS (Simulating your Project Logic)
    BATTERY_CRITICAL = 20  # If battery < 20%, always use small model
    
    # 2. ROUTING LOGIC
    if battery_percent < BATTERY_CRITICAL:
        selected_model = llm_small
        model_name = "Small Model (Battery Saver Mode)"
    elif is_complex_task:
        selected_model = llm_large
        model_name = "Large Model (High Performance Mode)"
    else:
        selected_model = llm_small
        model_name = "Small Model (Efficiency Mode)"

    print(f"--- Routing Decision: {model_name} ---")

    # 3. EXECUTE INFERENCE
    # We use a simple prompt format for the simulation
    output = selected_model(
        f"User: {user_prompt}\nAssistant:",
        max_tokens=64,
        stop=["User:", "\n"],
        echo=False
    )
    
    return output["choices"][0]["text"].strip()

# --- TEST THE SIMULATION ---

# Scenario 1: Low Battery (Should use Small Model)
print("Scenario 1: Low Battery")
print(run_smart_inference("Hi there!", battery_percent=15, is_complex_task=True))

print("\n" + "-"*30 + "\n")

# Scenario 2: High Battery + Hard Task (Should use Large Model)
print("Scenario 2: Complex Task")
print(run_smart_inference("Explain Quantum Physics in one sentence.", battery_percent=85, is_complex_task=True))

Scenario 1: Low Battery
--- Routing Decision: Small Model (Battery Saver Mode) ---


llama_perf_context_print:        load time =    1007.61 ms
llama_perf_context_print: prompt eval time =    1007.41 ms /     8 tokens (  125.93 ms per token,     7.94 tokens per second)
llama_perf_context_print:        eval time =     729.41 ms /    21 runs   (   34.73 ms per token,    28.79 tokens per second)
llama_perf_context_print:       total time =    1750.83 ms /    29 tokens
llama_perf_context_print:    graphs reused =         20


Hello! How can I help you today? Is there a particular topic or question you have in mind?

------------------------------

Scenario 2: Complex Task
--- Routing Decision: Large Model (High Performance Mode) ---


llama_perf_context_print:        load time =    2287.77 ms
llama_perf_context_print: prompt eval time =    1767.16 ms /    11 tokens (  160.65 ms per token,     6.22 tokens per second)
llama_perf_context_print:        eval time =   11133.84 ms /    47 runs   (  236.89 ms per token,     4.22 tokens per second)
llama_perf_context_print:       total time =   12929.06 ms /    58 tokens
llama_perf_context_print:    graphs reused =         46


Quantum Physics is a branch of physics that deals with the behavior of matter and energy at the atomic and subatomic level, and how it can exhibit both wave-particle duality and exhibit properties such as superposition and entanglement.


In [43]:
# --- MODULE C: THE INTELLIGENT ROUTER ---

def get_routing_decision(prompt, monitor):
    """
    Combines Hardware Status and Prompt Complexity to pick a model.
    """
    # 1. Use your existing TelemetryMonitor to get a snapshot 
    snap = monitor.snapshot()
    
    # 2. Use your existing SemanticClassifier logic for complexity [cite: 24, 233]
    # We initialize the classifier here to use its .classify() method
    classifier = SemanticClassifier()
    result = classifier.classify(prompt)
    
    # 3. Decision Logic [cite: 202]
    # Use your EPSILON_THRESHOLD (0.4) for battery efficiency [cite: 171]
    if snap.efficiency < EPSILON_THRESHOLD:
        return llm_small, f"Small Model (Power Save: Efficiency {snap.efficiency})"
    
    # Use your TAU_THRESHOLD (0.5) for semantic complexity [cite: 215]
    if result.score >= TAU_THRESHOLD:
        return llm_large, f"Large Model (High Performance: S(p)={result.score})"
    
    return llm_small, f"Small Model (Efficiency: Simple Task S(p)={result.score})"

# --- MODULE D: EXECUTION LOOP ---

def simulate_ai_assistant(user_input):
    # Use the correct class name from your file 
    monitor = TelemetryMonitor() 
    
    # Get Decision
    model, reason = get_routing_decision(user_input, monitor)
    
    print(f"Decision: {reason}")
    
    # Run Inference using your existing model variables [cite: 134]
    response = model(
        f"Q: {user_input} A:",
        max_tokens=50,
        stop=["Q:"],
        echo=False
    )
    
    print(f"Response: {response['choices'][0]['text'].strip()}")

# --- RUN THE TEST ---
test_prompt = "Explain the relationship between entropy and mutual information."
simulate_ai_assistant(test_prompt)

  Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4316.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing anchor embeddings ...
  SemanticClassifier ready in 14161 ms.

Decision: Large Model (High Performance: S(p)=0.5715)


llama_perf_context_print:        load time =    2287.77 ms
llama_perf_context_print: prompt eval time =    2137.06 ms /    13 tokens (  164.39 ms per token,     6.08 tokens per second)
llama_perf_context_print:        eval time =   11491.49 ms /    49 runs   (  234.52 ms per token,     4.26 tokens per second)
llama_perf_context_print:       total time =   13656.60 ms /    62 tokens
llama_perf_context_print:    graphs reused =         48


Response: Entropy is a measure of the uncertainty in a system, while mutual information is a measure of the dependence between two systems. As the amount of information about one system increases, the mutual information between the two systems decreases. Therefore, entropy and mutual information


In [44]:
# Fix 1: Initialize classifier ONCE outside the loop
classifier = SemanticClassifier()   # only this one instance

# Fix 2: Lower τ slightly to catch more hard queries
TAU_THRESHOLD   = 0.46
ROUTER_TAU      = 0.46

print("Fixes applied. Re-run your prompts loop now.")

  Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12877.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing anchor embeddings ...
  SemanticClassifier ready in 10142 ms.

Fixes applied. Re-run your prompts loop now.


In [45]:
prompts = [
    "Hi!",
    "What is 2+2?",
    "Explain the Jordan Canonical Form in detail.",
    "How does a cache hazard unit work?",
]

for p in prompts:
    decision = router.route(p)
    result   = executor.generate(p, decision)
    print(f"\nPrompt  : {p}")
    print(f"Model   : {decision.model.value}  |  S(p)={decision.complexity_score:.4f}  |  {result.tokens_generated} tokens  |  {result.total_ms:.0f}ms")


Prompt  : Hi!
Model   : M_small  |  S(p)=0.4444  |  108 tokens  |  136ms

Prompt  : What is 2+2?
Model   : M_small  |  S(p)=0.4478  |  64 tokens  |  81ms

Prompt  : Explain the Jordan Canonical Form in detail.
Model   : M_small  |  S(p)=0.4828  |  115 tokens  |  144ms

Prompt  : How does a cache hazard unit work?
Model   : M_large  |  S(p)=0.5117  |  186 tokens  |  744ms


In [46]:
import time
import json
import statistics
from dataclasses import dataclass, field
from typing import List
 
print("=" * 60)
print("  EVALUATION SCRIPT — Dynamic Inference System")
print("  Hardware-Aware LLM Routing — Paper Results Generator")
print("=" * 60)
print()
 
# ── Evaluation dataset ────────────────────────────────────────
# PAPER Section 4.1: We evaluate on 20 hand-labelled prompts
# spanning 4 difficulty categories. Ground truth labels were
# assigned by the authors based on expected reasoning depth.
#
# Label key:
#   "easy"  → correct answer is M_small
#   "hard"  → correct answer is M_large
 
EVAL_DATASET = [
    # ── Category 1: Trivial / conversational (expect M_small)
    {"prompt": "Hi, how are you?",                                          "ground_truth": "easy"},
    {"prompt": "What is 5 times 8?",                                        "ground_truth": "easy"},
    {"prompt": "What is the capital of Japan?",                             "ground_truth": "easy"},
    {"prompt": "Tell me a fun fact.",                                       "ground_truth": "easy"},
    {"prompt": "What does CPU stand for?",                                  "ground_truth": "easy"},
 
    # ── Category 2: Factual / moderate (expect M_small)
    {"prompt": "How does photosynthesis work?",                             "ground_truth": "easy"},
    {"prompt": "What is Python used for?",                                  "ground_truth": "easy"},
    {"prompt": "Explain what RAM does in a computer.",                      "ground_truth": "easy"},
    {"prompt": "What is the difference between HTTP and HTTPS?",            "ground_truth": "easy"},
    {"prompt": "What is machine learning in simple terms?",                 "ground_truth": "easy"},
 
    # ── Category 3: Technical / analytical (expect M_large)
    {"prompt": "Derive the backpropagation algorithm for an LSTM.",         "ground_truth": "hard"},
    {"prompt": "Explain the Jordan Canonical Form with a proof.",           "ground_truth": "hard"},
    {"prompt": "How does a cache hazard unit work in a CPU pipeline?",      "ground_truth": "hard"},
    {"prompt": "Compare BERT and GPT-3 architectures in depth.",            "ground_truth": "hard"},
    {"prompt": "Explain the CAP theorem and its distributed system trade-offs.", "ground_truth": "hard"},
 
    # ── Category 4: Research-level (expect M_large)
    {"prompt": "Prove that the square root of 2 is irrational.",            "ground_truth": "hard"},
    {"prompt": "Design a fault-tolerant distributed key-value store.",      "ground_truth": "hard"},
    {"prompt": "Analyze semiconductor supply chain risks for AI hardware.", "ground_truth": "hard"},
    {"prompt": "Derive the attention gradient for transformer fine-tuning.","ground_truth": "hard"},
    {"prompt": "Explain mutual information and its role in representation learning.", "ground_truth": "hard"},
]
 
print(f"  Evaluation dataset loaded: {len(EVAL_DATASET)} prompts")
print(f"  Easy (M_small expected): {sum(1 for d in EVAL_DATASET if d['ground_truth']=='easy')}")
print(f"  Hard (M_large expected): {sum(1 for d in EVAL_DATASET if d['ground_truth']=='hard')}")
print()
print("  Eval Cell 1 OK")
 

  EVALUATION SCRIPT — Dynamic Inference System
  Hardware-Aware LLM Routing — Paper Results Generator

  Evaluation dataset loaded: 20 prompts
  Easy (M_small expected): 10
  Hard (M_large expected): 10

  Eval Cell 1 OK


In [47]:
# Fix Module B: change cpu_percent interval from 1s to 0
# interval=0 returns instantaneous reading, no waiting

import psutil

# Monkey-patch the _read_cpu method
def _fast_read_cpu(self):
    return psutil.cpu_percent(interval=0)

TelemetryMonitor._read_cpu = _fast_read_cpu
print("Fix applied — cpu_percent interval set to 0.")

Fix applied — cpu_percent interval set to 0.


In [48]:
# Fix: ensure classifier is initialized once and reused
classifier = SemanticClassifier()   # single instance
router.classifier = classifier      # patch router to use it
router.reset_stats()
executor.stats = SessionStats()
executor.clear_history()

print("Fix applied. Classifier is now a single instance.")
print("Ready to re-run evaluation.")

  Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3291.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Computing anchor embeddings ...
  SemanticClassifier ready in 14377 ms.

Fix applied. Classifier is now a single instance.
Ready to re-run evaluation.


In [49]:
@dataclass
class EvalRecord:
    prompt:           str
    ground_truth:     str
    predicted_model:  str
    correct:          bool
    complexity_score: float
    efficiency_score: float
    routing_reason:   str
    tokens_generated: int
    tokens_per_sec:   float
    energy_j:         float
    routing_ms:       float
    inference_ms:     float
    total_ms:         float

print("=" * 60)
print("  Running evaluation — this will take several minutes")
print("=" * 60)
print()

router.reset_stats()
executor.stats = SessionStats()
executor.clear_history()
eval_records = []

# ── Warm-up run (not counted) ─────────────────────────────────
# Forces classifier and models into memory before timing starts
print("  Warming up classifier and models (not counted)...")
_ = router.route("Hello")
_ = executor.generate("Hello", _)
print("  Warm-up done. Starting timed evaluation.\n")

for i, item in enumerate(EVAL_DATASET):
    prompt       = item["prompt"]
    ground_truth = item["ground_truth"]

    print(f"  [{i+1:02d}/20] {prompt[:50]}{'...' if len(prompt)>50 else ''}")

    # Time ONLY the routing decision (Modules A+B+C)
    t_route_start = time.time()
    decision      = router.route(prompt)
    routing_ms    = (time.time() - t_route_start) * 1000

    # Time ONLY the inference (Module D)
    t_infer_start = time.time()
    result        = executor.generate(prompt, decision)
    inference_ms  = (time.time() - t_infer_start) * 1000

    total_ms = routing_ms + inference_ms

    predicted = "easy" if decision.model.value == "M_small" else "hard"
    correct   = (predicted == ground_truth)
    status    = "✓" if correct else "✗"

    print(f"         S(p)={decision.complexity_score:.4f} "
          f"→ {decision.model.value}  {status}  "
          f"({result.tokens_generated} tok, "
          f"route={routing_ms:.1f}ms, infer={inference_ms:.1f}ms)")

    eval_records.append(EvalRecord(
        prompt           = prompt,
        ground_truth     = ground_truth,
        predicted_model  = decision.model.value,
        correct          = correct,
        complexity_score = decision.complexity_score,
        efficiency_score = decision.efficiency_score,
        routing_reason   = decision.reason,
        tokens_generated = result.tokens_generated,
        tokens_per_sec   = result.tokens_per_sec,
        energy_j         = result.energy_estimate_j,
        routing_ms       = routing_ms,
        inference_ms     = inference_ms,
        total_ms         = total_ms,
    ))

print()
print(f"  Data collection complete — {len(eval_records)} records.")
print("  Eval Cell 2 OK")

  Running evaluation — this will take several minutes

  Warming up classifier and models (not counted)...
  Warm-up done. Starting timed evaluation.

  [01/20] Hi, how are you?
         S(p)=0.4389 → M_small  ✓  (95 tok, route=93.5ms, infer=119.7ms)
  [02/20] What is 5 times 8?
         S(p)=0.4523 → M_small  ✓  (57 tok, route=75.4ms, infer=71.5ms)
  [03/20] What is the capital of Japan?
         S(p)=0.4299 → M_small  ✓  (52 tok, route=79.0ms, infer=65.4ms)
  [04/20] Tell me a fun fact.
         S(p)=0.4602 → M_small  ✓  (57 tok, route=94.4ms, infer=72.0ms)
  [05/20] What does CPU stand for?
         S(p)=0.4431 → M_small  ✓  (100 tok, route=84.0ms, infer=126.0ms)
  [06/20] How does photosynthesis work?
         S(p)=0.4968 → M_small  ✓  (41 tok, route=86.5ms, infer=52.5ms)
  [07/20] What is Python used for?
         S(p)=0.4453 → M_small  ✓  (53 tok, route=97.0ms, infer=67.0ms)
  [08/20] Explain what RAM does in a computer.
         S(p)=0.4877 → M_small  ✓  (45 tok, route=84.7ms, i

In [50]:
print()
print("=" * 60)
print("  TABLE 1 — Routing Latency Overhead")
print("  (Module A + B + C combined, before inference starts)")
print("=" * 60)
 
routing_times  = [r.routing_ms for r in eval_records]
inference_times = [r.inference_ms for r in eval_records]
total_times    = [r.total_ms for r in eval_records]
 
small_records = [r for r in eval_records if r.predicted_model == "M_small"]
large_records = [r for r in eval_records if r.predicted_model == "M_large"]
 
print(f"""
  Metric                        Value
  ──────────────────────────── ──────────────
  Avg Routing Overhead         {statistics.mean(routing_times):.2f} ms
  Min Routing Overhead         {min(routing_times):.2f} ms
  Max Routing Overhead         {max(routing_times):.2f} ms
  Std Dev Routing              {statistics.stdev(routing_times):.2f} ms
 
  Avg Inference (M_small)      {statistics.mean([r.inference_ms for r in small_records]):.2f} ms
  Avg Inference (M_large)      {statistics.mean([r.inference_ms for r in large_records]):.2f} ms
 
  Avg Total Latency (M_small)  {statistics.mean([r.total_ms for r in small_records]):.2f} ms
  Avg Total Latency (M_large)  {statistics.mean([r.total_ms for r in large_records]):.2f} ms
 
  Routing Overhead %           {(statistics.mean(routing_times)/statistics.mean(total_times))*100:.2f}%
  (routing_ms / total_ms)
""")
print("  Eval Cell 3 OK")


  TABLE 1 — Routing Latency Overhead
  (Module A + B + C combined, before inference starts)

  Metric                        Value
  ──────────────────────────── ──────────────
  Avg Routing Overhead         125.21 ms
  Min Routing Overhead         75.40 ms
  Max Routing Overhead         209.98 ms
  Std Dev Routing              45.88 ms

  Avg Inference (M_small)      85.96 ms
  Avg Inference (M_large)      740.79 ms

  Avg Total Latency (M_small)  184.05 ms
  Avg Total Latency (M_large)  899.14 ms

  Routing Overhead %           24.75%
  (routing_ms / total_ms)

  Eval Cell 3 OK


In [51]:
print()
print("=" * 60)
print("  TABLE 2 — Routing Accuracy")
print("=" * 60)
 
total     = len(eval_records)
correct   = sum(1 for r in eval_records if r.correct)
incorrect = total - correct
 
# Per-category accuracy
easy_records  = [r for r in eval_records if r.ground_truth == "easy"]
hard_records  = [r for r in eval_records if r.ground_truth == "hard"]
easy_correct  = sum(1 for r in easy_records if r.correct)
hard_correct  = sum(1 for r in hard_records if r.correct)
 
# False positive / negative
# False Positive = routed to M_large when M_small was correct (wasteful)
# False Negative = routed to M_small when M_large was needed (quality loss)
false_pos = sum(1 for r in eval_records if r.ground_truth=="easy" and r.predicted_model=="M_large")
false_neg = sum(1 for r in eval_records if r.ground_truth=="hard" and r.predicted_model=="M_small")
 
print(f"""
  Metric                        Value
  ──────────────────────────── ──────────────
  Overall Accuracy             {correct}/{total}  ({100*correct/total:.1f}%)
  Easy Query Accuracy          {easy_correct}/{len(easy_records)}  ({100*easy_correct/len(easy_records):.1f}%)
  Hard Query Accuracy          {hard_correct}/{len(hard_records)}  ({100*hard_correct/len(hard_records):.1f}%)
 
  False Positives              {false_pos}  (easy→M_large, wastes energy)
  False Negatives              {false_neg}  (hard→M_small, quality loss)
 
  τ threshold used             {ROUTER_TAU}
  ε threshold used             {ROUTER_EPSILON}
""")
 
print("  Per-prompt accuracy breakdown:")
print(f"  {'Prompt':<45} {'GT':>5} {'Pred':>8} {'S(p)':>6} {'OK':>4}")
print("  " + "-" * 72)
for r in eval_records:
    gt_label   = "small" if r.ground_truth == "easy" else "large"
    pred_label = "small" if r.predicted_model == "M_small" else "large"
    status     = "✓" if r.correct else "✗"
    print(f"  {r.prompt[:44]:<45} {gt_label:>5} {pred_label:>8} "
          f"{r.complexity_score:>6.4f} {status:>4}")
 
print()
print("  Eval Cell 4 OK")
 


  TABLE 2 — Routing Accuracy

  Metric                        Value
  ──────────────────────────── ──────────────
  Overall Accuracy             19/20  (95.0%)
  Easy Query Accuracy          10/10  (100.0%)
  Hard Query Accuracy          9/10  (90.0%)

  False Positives              0  (easy→M_large, wastes energy)
  False Negatives              1  (hard→M_small, quality loss)

  τ threshold used             0.46
  ε threshold used             0.4

  Per-prompt accuracy breakdown:
  Prompt                                           GT     Pred   S(p)   OK
  ------------------------------------------------------------------------
  Hi, how are you?                              small    small 0.4389    ✓
  What is 5 times 8?                            small    small 0.4523    ✓
  What is the capital of Japan?                 small    small 0.4299    ✓
  Tell me a fun fact.                           small    small 0.4602    ✓
  What does CPU stand for?                      small    small 

In [53]:
print()
print("=" * 60)
print("  TABLE 3 — Energy Efficiency")
print("  Primary paper metric: Energy per 1,000 tokens")
print("=" * 60)
 
# ── Dynamic system energy (actual) ───────────────────────────
total_tokens_dynamic = sum(r.tokens_generated for r in eval_records)
total_energy_dynamic = sum(r.energy_j for r in eval_records)
 
# ── Baseline: always use M_large ─────────────────────────────
# Simulated by applying M_large TDP to all inference times
total_energy_baseline = sum(
    (TDP_LARGE_WATTS * (r.total_ms / 1000.0))
    for r in eval_records
)
 
# ── Baseline: always use M_small ─────────────────────────────
total_energy_small_only = sum(
    (TDP_SMALL_WATTS * (r.total_ms / 1000.0))
    for r in eval_records
)
 
# ── Per-1K token metrics ──────────────────────────────────────
e_per_1k_dynamic  = (total_energy_dynamic  / total_tokens_dynamic) * 1000
e_per_1k_baseline = (total_energy_baseline / total_tokens_dynamic) * 1000
e_per_1k_small    = (total_energy_small_only / total_tokens_dynamic) * 1000
 
energy_saved_vs_large = ((total_energy_baseline - total_energy_dynamic) /
                          total_energy_baseline) * 100
 
print(f"""
  Metric                        Dynamic    M_large    M_small
                                System     Always     Always
  ──────────────────────────── ────────── ────────── ──────────
  Total Energy (J)             {total_energy_dynamic:>9.4f}  {total_energy_baseline:>9.4f}  {total_energy_small_only:>9.4f}
  Energy / 1K Tokens (J)       {e_per_1k_dynamic:>9.4f}  {e_per_1k_baseline:>9.4f}  {e_per_1k_small:>9.4f}
  Total Tokens Generated       {total_tokens_dynamic:>9}
  M_small Queries              {len(small_records):>9}  ({100*len(small_records)/total:.1f}%)
  M_large Queries              {len(large_records):>9}  ({100*len(large_records)/total:.1f}%)
 
  Energy Saved vs M_large Only : {energy_saved_vs_large:.2f}%
  ← KEY PAPER RESULT (Section 5.2)
""")
print("  Eval Cell 5 OK")
 


  TABLE 3 — Energy Efficiency
  Primary paper metric: Energy per 1,000 tokens

  Metric                        Dynamic    M_large    M_small
                                System     Always     Always
  ──────────────────────────── ────────── ────────── ──────────
  Total Energy (J)              837.8732  1214.0239   404.6746
  Energy / 1K Tokens (J)        346.9454   502.7014   167.5671
  Total Tokens Generated            2415
  M_small Queries                     11  (55.0%)
  M_large Queries                      9  (45.0%)

  Energy Saved vs M_large Only : 30.98%
  ← KEY PAPER RESULT (Section 5.2)

  Eval Cell 5 OK
